# Эксперимент 2.4: Сравнение методов на ML задачах

In [1]:
import urllib.request
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.datasets import load_svmlight_file

from oracles import LogCoshL2Oracle, ExponentialLossL2Oracle
from optimization import nonlinear_conjugate_gradients, hessian_free_newton, lbfgs

In [ ]:
LIBSVM_BASE = "https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets"
data_dir = Path("data/libsvm")
data_dir.mkdir(parents=True, exist_ok=True)

def ensure_dataset(name, rel_url):
    path = data_dir / name
    if not path.exists():
        req = urllib.request.Request(f"{LIBSVM_BASE}/{rel_url}", headers={"User-Agent": "MetOpt-lab/2.0"})
        with urllib.request.urlopen(req, timeout=300) as r, open(path, "wb") as f:
            f.write(r.read())
    return path

def labels_pm_one(y):
    y = np.asarray(y, dtype=float).ravel()
    u = np.unique(y)
    if len(u) == 2 and 0.0 in u and 1.0 in u:
        return 2.0 * y - 1.0
    return y

def sparse_oracle_ops(X):
    X = X.tocsr().astype(np.float64)
    def matvec_Ax(x):
        return X.dot(np.asarray(x, dtype=np.float64).ravel())
    def matvec_ATx(v):
        return X.T.dot(np.asarray(v, dtype=np.float64).ravel())
    def matmat_ATsA(s):
        s = np.asarray(s, dtype=np.float64).ravel()
        return X.T.dot(sparse.diags(s).dot(X)).toarray()
    return matvec_Ax, matvec_ATx, matmat_ATsA

reg_path = ensure_dataset("abalone_scale", "regression/abalone_scale")
clf_path = ensure_dataset("a9a", "binary/a9a")
X_reg, y_reg = load_svmlight_file(reg_path)
X_clf, y_clf = load_svmlight_file(clf_path)
y_reg = np.asarray(y_reg, dtype=np.float64).ravel()
y_clf = labels_pm_one(y_clf)

m_reg, n_reg = X_reg.shape
m_clf, n_clf = X_clf.shape
oracle_reg = LogCoshL2Oracle(*sparse_oracle_ops(X_reg), y_reg, 1.0 / m_reg)
oracle_clf = ExponentialLossL2Oracle(*sparse_oracle_ops(X_clf), y_clf, 1.0 / m_clf)

In [ ]:
def run_methods(oracle, x0):
    ls = {'method': 'Wolfe', 'c1': 1e-4, 'c2': 0.9, 'alpha_0': 1.0}
    out = {}
    out['NLCG'] = nonlinear_conjugate_gradients(oracle, x0, tolerance=1e-8, max_iter=150, line_search_options=ls, trace=True)
    out['HFN'] = hessian_free_newton(oracle, x0, tolerance=1e-8, max_iter=80, line_search_options=ls, trace=True)
    out['L-BFGS(L=10)'] = lbfgs(oracle, x0, tolerance=1e-8, max_iter=150, memory_size=10, line_search_options=ls, trace=True)
    return out

x0_reg = np.zeros(n_reg)
x0_clf = np.zeros(n_clf)
res_reg = run_methods(oracle_reg, x0_reg)
res_clf = run_methods(oracle_clf, x0_clf)

In [ ]:
def draw_family(results, title):
    plt.figure(figsize=(15, 4))

    plt.subplot(1, 3, 1)
    for name, (_, _, h) in results.items():
        plt.plot(h['func'], label=name)
    plt.xlabel('iteration')
    plt.ylabel('f(x_k)')
    plt.title(title + ': f vs iter')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.subplot(1, 3, 2)
    for name, (_, _, h) in results.items():
        plt.plot(h['time'], h['func'], label=name)
    plt.xlabel('time, sec')
    plt.ylabel('f(x_k)')
    plt.title(title + ': f vs time')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.subplot(1, 3, 3)
    for name, (_, _, h) in results.items():
        g = np.array(h['grad_norm'])
        rel = (g ** 2) / max(g[0] ** 2, 1e-32)
        plt.plot(h['time'], rel, label=name)
    plt.yscale('log')
    plt.xlabel('time, sec')
    plt.ylabel(r'$||g_k||^2 / ||g_0||^2$')
    plt.title(title + ': rel grad vs time')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.tight_layout()

draw_family(res_reg, 'Regression')
draw_family(res_clf, 'Classification')